In [ ]:
link_api_data_sus = "https://apidadosabertos.saude.gov.br/v1/#/Agravo%20Arboviroses/get_arboviroses_dengue"

In [1]:
import os, sys, requests, pprint
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [2]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .getOrCreate()

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [9]:

url = "https://apidadosabertos.saude.gov.br/arboviroses/dengue"

limit = 20
offset = 0

todos = []

while True:
    try:
        response = requests.get(
            url,
            params={
                "nu_ano": 2026,
                "limit": limit,
                "offset": offset
            }
        )

        data = response.json()

        registros = data.get("parametros", [])

        if not registros or offset >= 200:
            break

        todos.extend(registros)

        msg = f"Offset={offset} | Recebidos={len(registros)}"
        print(msg)

        offset += limit
    except Exception as e:
        print("\t","Erro: ", str(e)[:300])



Offset=0 | Recebidos=20
Offset=20 | Recebidos=20
Offset=40 | Recebidos=20
Offset=60 | Recebidos=20
Offset=80 | Recebidos=20
Offset=100 | Recebidos=20
Offset=120 | Recebidos=20
Offset=140 | Recebidos=20
Offset=160 | Recebidos=20
Offset=180 | Recebidos=20


In [10]:


campos = registros[0].keys()

schema = StructType([
    StructField(campo, StringType(), True)
    for campo in campos
])

# Cria DataFrame Spark
df = spark.createDataFrame(todos, schema)

df.show(5, truncate=False)

+------+---------+----------+-------+------+---------+----------+----------+----------+----------+-------+----------+-------+----------+-------+----------+-----+----------+----------+-------+---------+----------+---------+---------+---------+----------+----------+----------+----------+---------+--------+----------+--------+--------+----------+----------+----------+--------+---------+-------+---------+--------+----------+--------+--------+----------+----------+--------+-------+-----+---------+--------+-------+------+----------+---------+----------+-------+--------+----------+----------+---+---------+------------+--------+-----+-------+--------+--------+------+------+----------+----------+-------+---------+----------+----------+----+---------+--------+---------+---------+-----+----------+----------+----------+----------+----------+-------+----------+----------+----------+------+---------+----------+----------+---------+--------+---------+----------+----------+----------+----------+------

In [5]:
# df.printSchema()
df.count()

40

In [ ]:
df.filter("id_municip = '330022'").count() #.show(100, truncate=False)

# df.select("tp_not", "dt_notific", "nu_ano", "sg_uf").show(5, truncate=False)


# (df.select("dt_notific"
#           ,"id_municip"
#           ,"ano_nasc" 
#           ,"cs_sexo"
#           ,"cs_raca"
#           )
#    .show(5, truncate=False))



In [8]:
import time
import pandas as pd
import requests


def carregar_dados_dengue(
    limit: int = 100, max_paginas: int = None
) -> pd.DataFrame:
    """Consome a API de Arboviroses/Dengue do Ministério da Saúde com paginação por offset.

    :param limit: Quantidade de registros por requisição (máx/padrão costuma
    ser 100)
    :param max_paginas: Limite opcional de páginas para baixar (None para buscar
    tudo)
    :return: DataFrame do Pandas com todos os registros acumulados
    """
    base_url = "https://apidadosabertos.saude.gov.br/arboviroses/dengue"
    offset = 0
    pagina = 1
    todos_registros = []

    # Opcional: defina um User-Agent limpo para evitar bloqueios de taxa
    headers = {"User-Agent": "Python/PaginationScript"}

    print("Iniciando coleta de dados...")

    while True:
        params = {"limit": limit, "offset": offset}

        try:
            response = \
                requests.get(base_url
                            ,params=params
                            # ,headers=headers
                            ,timeout=30
            )
            response.raise_for_status()
            dados = response.json()

            print("**** type(dados), len(dados)", type(dados), len(dados))
            print(dados)

            # A API costuma retornar uma lista direta de objetos ou uma chave contendo a lista
            if isinstance(dados, dict):
                # Se os dados vierem encapsulados (ex: dados['results'] ou dados['data'])
                registros = dados.get("results") or dados.get("data") or []
            elif isinstance(dados, list):
                registros = dados
            else:
                registros = []

            # Se não retornar mais registros, finalizamos a paginação
            if not registros:
                print("\nNenhum dado adicional retornado. Coleta concluída.")
                break

            todos_registros.extend(registros)
            print(
                f"Página {pagina} ok! {len(registros)} registros recuperados (Total acumulado: {len(todos_registros)})"
            )

            # Condição de parada manual (caso queira limitar o total de requisições)
            if max_paginas and pagina >= max_paginas:
                print(f"\nLimite de {max_paginas} páginas atingido.")
                break

            # Atualiza o offset e a contagem de páginas
            offset += limit
            pagina += 1

            # Pausa sutil de 0.2s para respeitar a infraestrutura da API pública
            time.sleep(0.2)

        except requests.exceptions.RequestException as e:
            print(f"\nErro ao requisitar offset={offset}: {e}")
            break

    df = pd.DataFrame(todos_registros)
    return df


if __name__ == "__main__":
    # Exemplo de uso: baixando as primeiras 5 páginas (500 registros)
    # Se quiser baixar TUDO, remova o parâmetro `max_paginas=5`
    df_dengue = carregar_dados_dengue(limit=100, max_paginas=5)

    print("\n--- Informações do DataFrame resultante ---")
    print(df_dengue.info())
    print(df_dengue.head())

    # Exemplo de salvamento local
    # df_dengue.to_csv("dengue_dados.csv", index=False)

Iniciando coleta de dados...
**** type(dados), len(dados) <class 'dict'> 1
{'parametros': [{'tp_not': '2', 'id_agravo': 'A90', 'dt_notific': '2025-03-03', 'sem_not': '202510', 'nu_ano': '2025', 'sg_uf_not': '16', 'id_municip': '160030', 'id_regiona': '0', 'id_unidade': '7709196', 'dt_sin_pri': '2025-02-27', 'sem_pri': '202509', 'nu_idade_n': '4063', 'cs_sexo': 'F', 'cs_gestant': '5', 'cs_raca': '4', 'cs_escol_n': '9', 'sg_uf': '16', 'id_mn_resi': '160030', 'id_rg_resi': '0', 'id_pais': '1', 'nduplic_n': 'nan', 'dt_digita': '2025-04-15', 'cs_flxret': '1', 'flxrecebi': 'nan', 'migrado_w': 'nan', 'dt_invest': '2025-03-03', 'id_ocupa_n': 'nan', 'dt_soro': '2025-03-03', 'resul_soro': 'nan', 'histopa_n': '4', 'dt_viral': 'nan', 'resul_vi_n': '4', 'sorotipo': 'nan', 'imunoh_n': '4', 'dt_pcr': 'nan', 'resul_pcr_': '4', 'classi_fin': '10', 'criterio': '2', 'tpautocto': '1', 'coufinf': '16', 'copaisinf': '1', 'comuninf': '160030', 'doenca_tra': 'nan', 'evolucao': '1', 'dt_obito': 'nan', 'dt_ence